# Tiny Transformer Playground

This notebook trains a very small Transformer model on a toy corpus and shows
live attention weights while training. All dependencies are standard PyPI
packages.


In [ ]:
!pip install -q torch matplotlib ipywidgets tqdm

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F, random, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display

## Hyper-parameters

In [ ]:
import ipywidgets as wd
cfg = {
    'emb':    wd.IntSlider(      32,  8,128,8 , description='Embed'),
    'heads':  wd.IntSlider(       2,  1,  8,1 , description='Heads'),
    'layers': wd.IntSlider(       2,  1,  6,1 , description='Layers'),
    'seq':    wd.IntSlider(      16,  4, 32,4 , description='SeqLen'),
    'bs':     wd.IntSlider(      32,  4,128,4 , description='Batch'),
    'steps':  wd.IntSlider(    1000,100,3000,100, description='Train steps')
}
display(*cfg.values())

## Toy corpus and dataloader

In [ ]:
text  = ('the quick brown fox jumps over the lazy dog ' * 200).split()
vocab = sorted(set(text)); stoi={w:i for i,w in enumerate(vocab)}; itos={i:w for w,i in stoi.items()}
vsz   = len(vocab)

def batch(bs, seq):
    while True:
        s = random.randint(0, len(text)-seq-2)
        ids = [stoi[w] for w in text[s:s+seq+1]]
        x,y = ids[:-1], ids[1:]
        yield torch.tensor(x), torch.tensor(y)


## Mini-Transformer implementation

In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, vsz, emb, heads, layers, seq):
        super().__init__()
        self.tok   = nn.Embedding(vsz, emb)
        self.pos   = nn.Parameter(torch.randn(seq, emb))
        self.blocks= nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=emb,
                                       nhead=heads,
                                       dim_feedforward=2*emb,
                                       batch_first=True,
                                       norm_first=True,
                                       dropout=0.0)
            for _ in range(layers)])
        self.ln_out= nn.LayerNorm(emb)
        self.head  = nn.Linear(emb, vsz, bias=False)
    def forward(self, x):
        B,T = x.shape
        h = self.tok(x) + self.pos[:T]
        for blk in self.blocks:         # save last block's attn map for visualization
            h, attn = blk.self_attn_output = blk.self_attn(h, h, h,
                                                           need_weights=True,
                                                           average_attn_weights=False)
        return self.head(self.ln_out(h)), attn           # attn: (B, heads, T, T)


## Training loop with live plots

In [ ]:
def train(cfg):
    device='cuda' if torch.cuda.is_available() else 'cpu'
    net = TinyTransformer(vsz, cfg['emb'].value,
                          cfg['heads'].value,
                          cfg['layers'].value,
                          cfg['seq'].value).to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3)
    gen = batch(cfg['bs'].value, cfg['seq'].value)
    
    # --- live plot set-up ---
    fig,(ax_loss,ax_attn) = plt.subplots(1,2, figsize=(10,4))
    losses,xdata = [],[]
    im = ax_attn.imshow(torch.zeros(cfg['seq'].value, cfg['seq'].value),
                        vmin=0, vmax=1, cmap='viridis')
    ttl = ax_attn.set_title('Attention')
    ln, = ax_loss.plot([],[], lw=2); ax_loss.set_ylabel('loss'); ax_loss.set_xlim(0,cfg['steps'].value)
    
    def _step(i):
        net.train(); x,y = next(gen); x,y = x.to(device), y.to(device)
        opt.zero_grad()
        logits, attn = net(x)
        loss = F.cross_entropy(logits.reshape(-1,vsz), y.flatten())
        loss.backward(); opt.step()
        # log
        losses.append(loss.item()); xdata.append(i)
        ln.set_data(xdata, losses); ax_loss.set_ylim(0,max(losses))
        # show last-head attention for first sample
        att = attn[0,-1].detach().cpu()
        im.set_data(att); ttl.set_text('Step %d  Head-%d'%(i,attn.shape[1]-1))
        return ln, im
    
    anim = animation.FuncAnimation(fig, _step, interval=50,
                                   frames=cfg['steps'].value, blit=False)
    plt.close(fig); display(anim)
    return net


In [ ]:
trained = train(cfg)

## Using the model interactively

In [ ]:
def predict(prompt, net, max_new=20):
    net.eval()
    toks = [stoi[w] for w in prompt.split()]
    for _ in range(max_new):
        x = torch.tensor(toks[-cfg['seq'].value:]).unsqueeze(0)
        logits,_ = net(x)
        next_id  = logits[0,-1].argmax().item()
        toks.append(next_id)
    return ' '.join(itos[i] for i in toks)

print(predict('the quick brown', trained, 10))
